# Qwen: kiểm tra bộ lọc gợi ý ngữ pháp trước và sau sửa

Notebook chỉ đọc các đầu ra Qwen đã lưu. Không gọi model, không train, không thay đổi transcript hay dùng câu sửa làm chuẩn phát âm.

Bộ quy tắc được sửa sau khi đã xem các câu này: đây là **kiểm tra hồi quy**, không phải benchmark mù. Tỷ lệ được hiển thị không phải độ chính xác ngữ pháp hay mức bảo toàn ý nghĩa.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display, HTML

ROOT = Path.cwd()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
OUT = ROOT / "data/generated/grammar-guard-20261024"
result = json.loads((OUT / "comparison.json").read_text(encoding="utf-8"))
manifest = json.loads((OUT / "report_manifest.json").read_text(encoding="utf-8"))
print("Report:", OUT / "report.html")
print("Generated:", result["generated_at_utc"])
print("Saved cases:", len(result["rows"]))

Report: D:\NCC\MDD\data\generated\grammar-guard-20261024\report.html
Generated: 2026-10-24T05:44:38.905740+00:00
Saved cases: 216


## 1. Kiểm tra nguồn

Checksum phải khớp dữ liệu, hai bộ kiểm tra và đầu ra báo cáo. Bộ kiểm tra cũ đã tái hiện mọi quyết định trong bản audit trước; các dự đoán gốc không được tạo lại.

In [2]:
def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

checks = []
for path, expected in manifest["inputs_sha256"].items():
    checks.append({"file": str(Path(path).relative_to(ROOT)), "matches": sha256(path) == expected})
for name, expected in manifest["outputs_sha256"].items():
    checks.append({"file": str((OUT / name).relative_to(ROOT)), "matches": sha256(OUT / name) == expected})
assert all(row["matches"] for row in checks), "Saved audit inputs or outputs changed; regenerate the report."
assert result["no_model_calls"] and result["no_training"]
assert manifest["baseline_reproduces_original_audit"]
display(pd.DataFrame(checks))

,file,matches
0,data\generated\model-comparison-20261024\gramm...,True
1,data\generated\model-comparison-20261024\gramm...,True
2,data\generated\model-comparison-20261024\gramm...,True
3,data\generated\model-comparison-20261024\gramm...,True
4,data\generated\grammar-guard-20261024\baseline...,True
5,data\generated\grammar-guard-20261024\baseline...,True
6,data\generated\grammar-guard-20261024\baseline...,True
7,scripts\sentence_rewrite.py,True
8,scripts\grammar_inference.py,True
9,scripts\compare_grammar_guards.py,True


## 2. Số gợi ý được hiển thị

Tách riêng 200 câu có đáp án công bố và 16 lượt kiểm tra phát sinh từ người dùng. Câu giữ nguyên, chỉnh dấu câu và chỉnh từ được đếm riêng. Một gợi ý được bộ lọc chấp nhận vẫn cần được xem là đề xuất.

In [3]:
summaries = []
for cohort, tasks in result["summary"].items():
    for task, values in tasks.items():
        summaries.append({
            "cohort": cohort, "task": task, "rows": values["rows"],
            "delivered_before": values["baseline"]["delivered"],
            "delivered_after": values["current"]["delivered"],
            "newly_accepted": values["newly_accepted"],
            "newly_withheld": values["newly_withheld"],
            "unchanged_after": values["current"]["unchanged_delivered"],
            "format_only_after": values["current"]["format_only_delivered"],
            "lexical_change_after": values["current"]["lexical_changes_delivered"],
            "reference_matches_withheld_before": values["baseline"]["reference_matches_withheld"],
            "reference_matches_withheld_after": values["current"]["reference_matches_withheld"],
        })
display(pd.DataFrame(summaries))

,cohort,task,rows,delivered_before,delivered_after,newly_accepted,newly_withheld,unchanged_after,format_only_after,lexical_change_after,reference_matches_withheld_before,reference_matches_withheld_after
0,published_heldout,grammar_corrected,100,68,73,5,0,51,6,16,7,5
1,published_heldout,natural,100,45,47,3,1,26,5,16,7,5
2,user_development,grammar_corrected,8,7,7,0,0,3,1,3,0,0
3,user_development,natural,8,7,7,0,0,3,1,3,0,0


## 3. Gợi ý trùng đáp án công bố được khôi phục

Chỉ so khớp chính xác sau khi chuẩn hóa khoảng trắng. Trùng đáp án không chứng minh câu hoàn chỉnh hay đúng ý người nói; một cách viết khác đáp án cũng có thể đúng.

In [4]:
recovered = [row for row in result["rows"]
             if row["matches_published_reference"] is True and row["transition"] == "newly_accepted"]
display(pd.DataFrame([{key: row[key] for key in ("id", "task", "source", "prediction")} for row in recovered]))
print("Reference-matching suggestions recovered:", len(recovered))

,id,task,source,prediction
0,wi:A:train:4525:a0,grammar_corrected,I studied in Al Qassim University and my major...,I studied at Al Qassim University and my major...
1,wi:C:train:8790:a0,grammar_corrected,"I was still out of breath sometimes, but I kep...","I was still out of breath sometimes, but I kep..."
2,jfleg:test:547:a0,natural,"While studying at university, people may still...","While studying at university, people may still..."
3,jfleg:test:282:a0,natural,Young people can work easily and can search an...,Young people can work easily and can search fo...


Reference-matching suggestions recovered: 4


## 4. Những gợi ý mới bị chặn

Cần xem riêng các gợi ý này: giảm số gợi ý không tự chứng minh bộ lọc an toàn hơn. Cột chẩn đoán mô tả quy tắc đã được áp dụng.

In [5]:
withheld = [row for row in result["rows"] if row["transition"] == "newly_withheld"]
display(pd.DataFrame([{
    "id": row["id"], "task": row["task"], "source": row["source"], "prediction": row["prediction"],
    "matches_reference": row["matches_published_reference"],
    "diagnostics": row["current"]["diagnostics"],
} for row in withheld]))
print("Newly withheld:", len(withheld))

,id,task,source,prediction,matches_reference,diagnostics
0,jfleg:test:260:a0,natural,"When someone did a legal behavior, he has to j...","When someone does a legal behavior, he has to ...",False,{'guard_validation': {'grammar_corrected': {'a...


Newly withheld: 1


## 5. Mọi quyết định thay đổi

Cột `prediction` là đúng văn bản Qwen đã sinh trước đó, không được bộ so sánh viết lại. Các gợi ý ngữ pháp và tự nhiên được kiểm tra độc lập.

In [6]:
transitions = [row for row in result["rows"] if row["transition"] in {"newly_accepted", "newly_withheld"}]
pd.set_option("display.max_colwidth", 180)
display(pd.DataFrame([{
    "id": row["id"], "cohort": row["cohort"], "task": row["task"],
    "transition": row["transition"], "source": row["source"], "prediction": row["prediction"],
    "matches_reference": row["matches_published_reference"], "diagnostics": row["current"]["diagnostics"],
} for row in transitions]))

,id,cohort,task,transition,source,prediction,matches_reference,diagnostics
0,wi:A:train:4525:a0,published_heldout,grammar_corrected,newly_accepted,I studied in Al Qassim University and my major Microbiology.,I studied at Al Qassim University and my major was Microbiology.,True,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': ['study_institution_p..."
1,wi:C:train:3056:a0,published_heldout,grammar_corrected,newly_accepted,"I'm completely sure that 50 years from now we will still be living on Earth, although our life styles will have changed a lot.","I'm completely sure that 50 years from now we will still be living on Earth, although our lifestyles will have changed a lot.",False,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': []}, 'natural': {'acc..."
2,wi:C:train:8790:a0,published_heldout,grammar_corrected,newly_accepted,"I was still out of breath sometimes, but I kept doing it because I wanted to achieve my goal to lose 10 pounds.","I was still out of breath sometimes, but I kept doing it because I wanted to achieve my goal of losing 10 pounds.",True,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': ['goal_gerund_complem..."
3,wi:B:train:4995:a0,published_heldout,grammar_corrected,newly_accepted,Michael can't believe of what he had just heard.,Michael can't believe what he had just heard.,False,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': ['redundant_complemen..."
4,wi:A:train:7268:a0,published_heldout,grammar_corrected,newly_accepted,My best skills are dealing with people as well as I am able do two things at the same time.,My best skills are dealing with people as well as I am able to do two things at the same time.,False,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': ['missing_infinitive_..."
5,jfleg:test:547:a0,published_heldout,natural,newly_accepted,"While studying at university, people may still do not know which subject they should specialize on.","While studying at university, people may still not know which subject they should specialize in.",True,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': []}, 'natural': {'acc..."
6,jfleg:test:282:a0,published_heldout,natural,newly_accepted,Young people can work easily and can search any information in a few minutes.,Young people can work easily and can search for any information in a few minutes.,True,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': []}, 'natural': {'acc..."
7,jfleg:test:260:a0,published_heldout,natural,newly_withheld,"When someone did a legal behavior, he has to justified by law.","When someone does a legal behavior, he has to be justified by law.",False,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': []}, 'natural': {'acc..."
8,jfleg:test:270:a0,published_heldout,natural,newly_accepted,It mentions about the cane toad which threatens many local species in Australia.,It mentions the cane toad which threatens many local species in Australia.,False,"{'guard_validation': {'grammar_corrected': {'accepted': True, 'reason': 'bounded_edits', 'version': 'bounded-grammatical-edits-v2', 'licensed_edit_types': []}, 'natural': {'acc..."


## 6. Giới hạn và file kết quả

Báo cáo HTML hiển thị từng quyết định thay đổi, câu gốc, đầu ra model và đáp án tham khảo. CSV/JSON chứa mọi lượt, kể cả những lượt không đổi quyết định.

In [7]:
for item in result["limitations"]:
    print("•", item)
for name in ("report.html", "comparison.csv", "comparison.json", "report_manifest.json"):
    print(OUT / name)

• Only the delivery validator changes; saved model predictions are identical in both arms.
• These examples were already observed before the rules were changed. This is a regression audit, not an unseen blind benchmark.
• Acceptance is not a grammar or semantic correctness label, and an unchanged sentence is not an improvement.
• Exact published-reference equality uses whitespace normalization only. Reference-matching does not prove a sentence is complete or intended.
• Nonmatching outputs may be valid alternatives. This audit cannot estimate semantic false-accept rates from reference matching alone.
• Published cases and the eight user-development examples per task are reported separately.
• The original frozen benchmark generation budget differs from the live worker. No new live inference latency is measured.
• Grammar and natural suggestions are checked independently and never become pronunciation references.
D:\NCC\MDD\data\generated\grammar-guard-20261024\report.html
D:\NCC\MDD\da